In [9]:
import os
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory
from datapizzai.type import TextBlock

# Carica variabili da .env (root progetto)
load_dotenv()

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1
)

# Invoke sempplice (minimale)
print(client.invoke("Ciao, piacere di conoscerti").text)

# Oppure usando un modulo che vedremo tra un attimo
print(client.invoke(TextBlock(content="Ciao, piacere di conoscerti")).text)

Ciao! Piacere mio. Come posso aiutarti oggi?
Ciao! Piacere di conoscerti anche per me. Sono il tuo assistente: come posso aiutarti oggi?


In [22]:
from datapizzai.cache import MemoryCache
import time

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1,
    cache=MemoryCache(),  # cache in-memory
)

# Stessa richiesta 2 volte: la seconda dovrebbe colpire la cache
q = "Dimmi 3 vantaggi del TDD in 1 riga"

t0 = time.perf_counter()
r1 = client.invoke(q)
t1 = time.perf_counter()
print("prima:", r1.text)
print(f"⏱️ tempo (prima): {t1 - t0:.3f}s")

t2 = time.perf_counter()
r2 = client.invoke(q)  # Qui avviene un cache hit, il client non viene invocato
t3 = time.perf_counter()
print("seconda:", r2.text)
print(f"⏱️ tempo (seconda): {t3 - t2:.3f}s")

prima: Feedback rapido; design più pulito e manutenibile; riduzione delle regressioni.
⏱️ tempo (prima): 6.340s
INFO     [2025-08-28 17:40:59 - datapizzai.cache.cache:109] Cache hit for 549d7d9fea78e1487f6a5993a57e7e217f9fd963d60bfb58febfbc0028fd5dec
seconda: Feedback rapido; design più pulito e manutenibile; riduzione delle regressioni.
⏱️ tempo (seconda): 0.000s


In [16]:
import os
from datapizzai.clients import ClientFactory
from datapizzai.memory import Memory
from datapizzai.type import TextBlock, ROLE

class Chatbot:
    def __init__(self, client, window_size: int = 6):
        self.client = client
        self.memory = Memory()
        self.window_size = window_size

    def _apply_sliding_window(self):
        if len(self.memory.memory) > self.window_size:
            self.memory.memory = self.memory.memory[-self.window_size:]

    def send(self, user_input: str) -> str:
        self.memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
        self._apply_sliding_window()
        response = self.client.invoke("", memory=self.memory)
        self.memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        # Stampa metriche minime (opzionale)
        total_tokens = (response.prompt_tokens_used or 0) + (response.completion_tokens_used or 0)
        print(f"[metriche] token totali: {total_tokens}")
        return response.text

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1,
)

bot = Chatbot(client, window_size=6)
print("Chat pronta. Digita 'esci' per terminare.")
while True:
    try:
        user = input("tu> ").strip()
        if user.lower() in {"esci", "exit", "quit"}:
            break
        print("bot>", bot.send(user))
    except KeyboardInterrupt:
        break
    except Exception:
        print("bot> Si è verificato un errore temporaneo. Riprova.")

Chat pronta. Digita 'esci' per terminare.


tu>  bella broooo


[metriche] token totali: 360
bot> Bellaaa! Dimmi pure, in cosa posso darti una mano? Preferisci italiano o inglese?


tu>  Ma quanto sei forte?


[metriche] token totali: 393
bot> Dipende da cosa ti serve! Sono “forte” nel:
- Spiegare concetti complessi in modo semplice
- Riassumere e tradurre testi
- Scrivere, correggere e migliorare contenuti
- Brainstorming di idee creative
- Aiutare con coding e debug

Niente panca piana però! Su cosa vuoi mettermi alla prova?


tu>  exit


In [19]:
from datapizzai.clients import ClientFactory
from datapizzai.type import TextBlock, MediaBlock, Media
from dotenv import load_dotenv
import os

load_dotenv('../.env')
client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o"
)

# Analizza immagine da URL
media = Media(
    extension="jpg",        # Estensione senza punto per MIME type corretto
    media_type="image",     # Tipo di media (image, audio, video)
    source_type="url",      # Fonte: url, base64, o file
    source="https://assets.science.nasa.gov/dynamicimage/assets/science/psd/mars/internal_resources/1155.jpeg?w=1767&h=350&fit=clip&crop=faces%2Cfocalpoint"
)

# Combina testo e immagine per input multimodale
response = client.invoke([
    TextBlock(content="Descrivi questa immagine in dettaglio"),
    MediaBlock(media=media)  # Wrapper che contiene l'immagine
])

print(response.text)

L'immagine è un panorama di un paesaggio marziano, probabilmente catturato da un rover della NASA. Presenta una vasta distesa di terreno roccioso e sabbioso con diverse formazioni geologiche. Sono visibili delle etichette che identificano particolari aree o punti di interesse, come "delta front", "Faillefeu", "Santa Cruz", "Marte" e "Rover Route".

Il terreno appare irregolare, con rocce sparse di varie dimensioni. L'orizzonte è leggermente ondulato e ci sono leggere variazioni di colore nel terreno, suggerendo differenti composizioni minerali o geologiche. L'immagine è suddivisa in sezioni numerate, probabilmente utilizzate per orientamento o riferimento scientifico. Lo sfondo è nero, evidenziando la superficie marziana.


In [21]:
import base64
from pathlib import Path
from datapizzai.type import Media, MediaBlock, TextBlock

def load_image_as_base64(path: str) -> str:
    """Converte file immagine in stringa base64 per trasmissione sicura"""
    return base64.b64encode(Path(path).read_bytes()).decode("utf-8")

# Carica immagine locale e converti in base64
image_b64 = load_image_as_base64("Client/Example.png")

# Crea oggetto Media con metadati dell'immagine
media = Media(
    extension="jpg",        # Estensione file per MIME type
    media_type="image",     # Tipo di contenuto
    source_type="base64",   # Formato di trasmissione
    source=image_b64,       # Dati immagine codificati
    detail="high"           # Qualità analisi (high per dettagli)
)

# Prompt specifico per analisi tecnica
prompt = "Analizza questa immagine e dammi una descrizione tecnica."

# Invoca AI con input multimodale (testo + immagine)
response = client.invoke([
    TextBlock(content=prompt),
    MediaBlock(media=media)  # Wrapper per l'immagine
])

print(response.content)

[TextBlock(content=L'immagine mostra un gruppo di persone sedute attorno a un tavolo, ciascuna con un laptop o un computer desktop. Sono impegnate in attività lavorative, probabilmente legate all'informatica o allo sviluppo software. Sul tavolo ci sono diverse lattine di bevande energetiche e una scatola di pizza aperta, con diverse fette già servite su piatti di carta. Ci sono anche fogli sparsi e post-it, suggerendo un ambiente di lavoro collaborativo e dinamico. L'illuminazione è fornita da una lampada da tavolo e l'atmosfera sembra serale o notturna. Su uno dei laptop è visibile il logo di Fedora, un sistema operativo basato su Linux.)]


In [27]:
from pathlib import Path
from datapizzai.type import Media, MediaBlock, TextBlock

analysis_client_google = client = ClientFactory.create(
            model="gemini-2.5-flash",
            provider="google",
            api_key=os.getenv("GOOGLE_API_KEY"),
            system_prompt="Sei un assistente AI esperto nell'analisi di audio. Rispondi in italiano.",
            temperature=0.5
        )

media = Media(
    extension="wav",
    media_type="audio",
    source_type="path",
    source="Client/TI0TpOD_.wav"        # <— percorso al file
)

prompt = "Trascrivi questo audio e riassumi il contenuto principale."
response = analysis_client_google.invoke([TextBlock(content=prompt), MediaBlock(media=media)])
print(response.content)

[TextBlock(content=Ecco la trascrizione e il riassunto dell'audio:

**Trascrizione:**
"Hello. We're excited to announce the new framework Data Piz AI."

**Riassunto del contenuto principale:**
L'audio annuncia con entusiasmo il lancio di un nuovo framework chiamato "Data Piz AI".)]


In [32]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv

from datapizzai.clients import ClientFactory
from datapizzai.memory import Memory
from datapizzai.type import ROLE, TextBlock, Media, MediaBlock

# Carica variabili d'ambiente
load_dotenv('../.env')

def create_mediablock_from_file(file_path: str) -> MediaBlock:
    """Crea un MediaBlock da un file immagine locale (base64)."""
    data = Path(file_path).read_bytes()
    image_b64 = base64.b64encode(data).decode('utf-8')
    ext = Path(file_path).suffix.lstrip('.').lower() or 'png'
    media = Media(
        extension=ext,
        media_type="image",
        source_type="base64",
        source=image_b64,
        detail="high",
    )
    return MediaBlock(media=media)

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o",
)
memory = Memory()

# Primo turno: utente invia immagine con richiesta
image_block = create_mediablock_from_file("Client/Example.png")
memory.add_turn([TextBlock("Analizza questa foto, cosa vedi?"), image_block], ROLE.USER)
resp = client.invoke("", memory=memory)
memory.add_turn([TextBlock(resp.text)], ROLE.ASSISTANT) #Aggiungo risposta alla memoria

# Secondo turno: follow-up che si basa sull'immagine precedente
memory.add_turn([TextBlock("Quali miglioramenti consiglieresti?")], ROLE.USER)
resp = client.invoke("", memory=memory)

print(resp.text)

[TextBlock(content=Per migliorare l'immagine e l'ambiente generale, potresti considerare i seguenti suggerimenti:

1. **Organizzazione del Tavolo**: Sistemare i cavi e rimuovere eventuali oggetti non necessari per un aspetto più ordinato.

2. **Illuminazione**: Aggiungere più sorgenti di luce calda per rendere l'ambiente più accogliente e ridurre l'affaticamento visivo.

3. **Comfort**: Assicurarsi che le sedie siano ergonomiche per migliorare il comfort durante le lunghe ore di lavoro.

4. **Spazio**: Aumentare lo spazio tra i partecipanti per una maggiore comodità e mobilità.

5. **Piante**: Aggiungere qualche pianta per migliorare l'atmosfera e la qualità dell'aria.

6. **Pause Regolari**: Introdurre momenti di pausa per migliorare la produttività e il benessere del team.

Questi suggerimenti possono contribuire a creare un ambiente di lavoro più piacevole ed efficiente.)]
